In [1]:
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.7 MB/s eta 0:00:00


In [2]:
import torch
from PIL import Image
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
import torch.nn as nn

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Initialize the model, preprocessing function and tokenizer

In [4]:
from clip_zeroshot import build_and_cache_text_features, build_and_cache_image_features, top_k_accuracy, load_cached_features

In [5]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
model.eval()  # model in train mode by default
model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-16')

open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


### Download and prepare the caltech data

In [6]:
caltech_dataset = datasets.Caltech101(
    root='./data',
    download=True,
    transform=preprocess
)

100%|██████████| 137M/137M [00:01<00:00, 76.0MB/s]


In [7]:
len(caltech_dataset)

8677

In [8]:
print(caltech_dataset.categories[:50])

['Faces', 'Faces_easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass', 'beaver', 'binocular', 'bonsai', 'brain', 'brontosaurus', 'buddha', 'butterfly', 'camera', 'cannon', 'car_side', 'ceiling_fan', 'cellphone', 'chair', 'chandelier', 'cougar_body', 'cougar_face', 'crab', 'crayfish', 'crocodile', 'crocodile_head', 'cup', 'dalmatian', 'dollar_bill', 'dolphin', 'dragonfly', 'electric_guitar', 'elephant', 'emu', 'euphonium', 'ewer', 'ferry', 'flamingo', 'flamingo_head', 'garfield', 'gerenuk', 'gramophone', 'grand_piano', 'hawksbill', 'headphone', 'hedgehog']


In [9]:
# Prepare the class names
caltech_class_names = [cls.replace('_', ' ') for cls in caltech_dataset.categories]
print(caltech_class_names[:10])

['Faces', 'Faces easy', 'Leopards', 'Motorbikes', 'accordion', 'airplanes', 'anchor', 'ant', 'barrel', 'bass']


In [10]:
# Create a dictionary of {class: [index_of_item1, index_of_item2....]}
from collections import defaultdict

total_labels = set()
class_to_indices = defaultdict(list)

for idx in range(len(caltech_dataset)):
    _, label = caltech_dataset[idx]
    class_to_indices[label].append(idx)
    total_labels.add(label)

In [11]:
# Sample 16 images for every class
import random
random.seed(42)

few_shot_indices = []

for label, indices in class_to_indices.items():
   sampled = random.sample(indices, min(16, len(indices)))
   few_shot_indices.extend(sampled)

In [12]:
# Build the training set
from torch.utils.data import Subset

few_shot_training_dataset = Subset(caltech_dataset, few_shot_indices)

In [13]:
# Build the evaluation set
all_indices = set(range(len(caltech_dataset)))
eval_indices = list(all_indices - set(few_shot_indices))
eval_dataset = Subset(caltech_dataset, eval_indices)

In [14]:
print(len(few_shot_training_dataset))
print(len(eval_dataset))

1616
7061


In [30]:
raw_train_loader = DataLoader(few_shot_training_dataset, batch_size=32, shuffle = True)

### Build the prompt learner

In [16]:
tokens = tokenizer(["a photo of a dog"]).to(device)
print(tokens.shape)   # confirm: token IDs, e.g. (1, 77)

embedded = model.token_embedding(tokens)   # try the layer you found
print(embedded.shape)  # should be (1, 77, embed_dim) — e.g. (1, 77, 512)

torch.Size([1, 77])
torch.Size([1, 77, 512])


In [24]:
class PromptLearner(nn.Module):
  def __init__(self, clip_model, n_ctx, tokenizer, ctx_dim, class_names):
    super().__init__()
    placeholder = "X " * n_ctx
    prompts = [f"{placeholder}{name}." for name in class_names]
    tokenized_prompts = tokenizer(prompts).to(device)
    self.num_classes = len(class_names)
    with torch.no_grad():
      embedding = clip_model.token_embedding(tokenized_prompts)

    prefix = embedding[:, :1, :]
    suffix = embedding[:, 1 + n_ctx:, :]

    self.register_buffer("prefix", prefix)
    self.register_buffer("suffix", suffix)
    self.register_buffer("tokenized_prompts", tokenized_prompts)
    self.ctx = nn.Parameter(torch.randn(n_ctx, ctx_dim) * 0.02)

  def forward(self):
    ctx = self.ctx.unsqueeze(0).expand(self.num_classes, -1, -1)
    prompts = torch.cat([self.prefix, ctx, self.suffix], dim=1)
    return prompts, self.tokenized_prompts

In [25]:
pl = PromptLearner(model, 4, tokenizer, 512, caltech_class_names).to(device)
prompts, tok_prompts = pl()

In [26]:
prompts.shape

torch.Size([101, 77, 512])

### Build the TextEncoderWrapper

In [27]:
from open_clip.transformer import text_global_pool

class TextEncoderWrapper(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.attn_mask = clip_model.attn_mask
        self.text_pool_type = clip_model.text_pool_type
        self.text_eos_id = getattr(clip_model, "text_eos_id", None)

    def forward(self, prompt_embeddings, tokenized_prompts):
        cast_dtype = self.transformer.get_cast_dtype()
        x = prompt_embeddings.to(cast_dtype) + self.positional_embedding.to(cast_dtype)
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)
        x = text_global_pool(x, tokenized_prompts, self.text_pool_type, eos_token_id=self.text_eos_id)
        if self.text_projection is not None:
            if isinstance(self.text_projection, nn.Linear):
                x = self.text_projection(x)
            else:
                x = x @ self.text_projection
        return x

In [28]:
text_encoder = TextEncoderWrapper(model)

text_features = text_encoder(prompts, tok_prompts)

In [29]:
text_features.shape

torch.Size([101, 512])

### Build the image features

In [32]:
train_image_cached = build_and_cache_image_features(model, device, raw_train_loader, './features', 'caltech-101-image')

  0%|          | 0/51 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/caltech-101-image.pt


In [34]:
from torch.utils.data import TensorDataset

image_feature_dataset = TensorDataset(train_image_cached["image_features"], train_image_cached["labels"])
train_loader = DataLoader(image_feature_dataset, batch_size=32, shuffle=True)

### Training Loop

In [35]:
for param in model.parameters():
    param.requires_grad_(False)

# PromptLearner's ctx should be the ONLY trainable thing
trainable = [name for name, p in pl.named_parameters() if p.requires_grad]
print(trainable)                    # expect just ['ctx']
print(pl.ctx.requires_grad)   # True

['ctx']
True


In [44]:
from tqdm.notebook import tqdm
import torch.nn.functional as F

epochs = 10
optimizer = torch.optim.SGD(pl.parameters(), lr=0.002)
num_ctx = 4
ctx_dim = 512
logit_scale = model.logit_scale.exp()

for epoch in range(epochs + 1):
  total_loss = 0
  for img_features, labels in tqdm(train_loader):
    img_features = img_features.to(device)
    labels = labels.to(device)

    prompts, tok_prompts = pl()
    text_features = text_encoder(prompts, tok_prompts)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    logits = logit_scale * img_features @ text_features.t()
    loss = F.cross_entropy(logits, labels)
    total_loss += loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  epoch_loss = total_loss / len(train_loader)
  print(f"epoch: {epoch + 1}, loss: {epoch_loss:.4f}")

  0%|          | 0/51 [00:00<?, ?it/s]

epoch: 1, loss: 0.1619


  0%|          | 0/51 [00:00<?, ?it/s]

epoch: 2, loss: 0.1578


  0%|          | 0/51 [00:00<?, ?it/s]

...

epoch: 9, loss: 0.1348


  0%|          | 0/51 [00:00<?, ?it/s]

epoch: 10, loss: 0.1316


  0%|          | 0/51 [00:00<?, ?it/s]

epoch: 11, loss: 0.1317


### Evaluation
Test accuracy: 90.21

In [47]:
eval_raw_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

eval_cached = build_and_cache_image_features(model, device, eval_raw_loader, './features', "caltech_eval")

eval_feature_dataset = TensorDataset(eval_cached["image_features"], eval_cached["labels"])
test_loader = DataLoader(eval_feature_dataset, batch_size=32, shuffle=False)

  0%|          | 0/221 [00:00<?, ?it/s]

Image features and labels has been saved at ./features/caltech_eval.pt


In [49]:
print([n for n, p in pl.named_parameters() if p.requires_grad])  # ['ctx'] only
print(sum(p.requires_grad for p in model.parameters()))  # 0

['ctx']
0


In [51]:
pl.eval()
with torch.no_grad():
    # 1. get the TRAINED text features — ctx is now learned
    prompts, tokenized = pl()
    text_features = text_encoder(prompts, tokenized)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # 2. eval over the TEST set's cached image features
    correct = 0
    total = 0
    for image_features, labels in test_loader:   # cached test features
        image_features = image_features.to(device)
        labels = labels.to(device)

        logits = image_features @ text_features.t()
        preds = logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}")

Test accuracy: 90.21


### Test Zero Shot CLIP
Zero-shot test accuracy: 84.58

In [56]:
clip_text_features = build_and_cache_text_features(model=model, tokenizer=tokenizer, classnames=caltech_class_names, templates=['a photo of a {}.'], device=device, cache_dir='./features', file_name='caltech_text_features_clip')

  0%|          | 0/101 [00:00<?, ?it/s]

Text features has been saved at ./features/caltech_text_features_clip.pt


In [57]:
clip_text_features = clip_text_features.to(device)

In [66]:
clip_text_features = clip_text_features.to(device)
clip_image_features = eval_cached['image_features'].to(device)
clip_labels = eval_cached['labels'].to(device)

correct = 0
total = 0

similarity = clip_image_features @ clip_text_features
preds = similarity.argmax(dim=-1)
correct += (clip_labels == preds).sum().item()
total += clip_labels.size(0)

accuracy = 100 * correct / total
print(f"Zero-shot test accuracy: {accuracy:.2f}")

Zero-shot test accuracy: 84.58
